In [0]:
%python
from pyspark.sql.functions import col, current_timestamp

In [0]:
%python
df_bronze = spark.table("bronze_investimentos_alpha")

In [0]:
%python
# 1. Removemos duplicados (Data + Ticker deve ser único)
df_silver = df_bronze.dropDuplicates(["date", "ticker"])

In [0]:
%python
display(df_silver)

In [0]:
%python
# 2. Garantimos que não existam preços vazios (nulos)
# Na Silver, o dado precisa estar pronto para análise
df_silver = df_silver.filter(col("close").isNotNull())

In [0]:
%python
df_silver = df_silver.withColumn("data_processamento", current_timestamp())

In [0]:
%python
nome_tabela_silver = "silver_investimentos"

df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(nome_tabela_silver)

print(f"Camada Silver finalizada com sucesso: {nome_tabela_silver}")
display(df_silver)

In [0]:
%python
# 1. Definição do Dicionário (Fácil de manter)
dicionario_investimentos = {
    "date": "Data do pregão em que os valores foram registrados (UTC)",
    "close": "Preço de fechamento do ativo ao final do dia",
    "high": "Maior preço atingido durante o pregão",
    "low": "Menor preço atingido durante o pregão",
    "open": "Preço de abertura do ativo no início do dia",
    "volume": "Quantidade total de títulos negociados no dia (Liquidez)",
    "ticker": "Código de identificação do ativo na bolsa (ex: PETR4.SA)",
    "data_processamento": "Data e hora em que o pipeline da Silver processou este registro"
}

# 2. Automação: Aplicando o dicionário diretamente nos metadados da tabela Delta
for coluna, comentario in dicionario_investimentos.items():
    try:
        spark.sql(f"ALTER TABLE {nome_tabela_silver} ALTER COLUMN {coluna} COMMENT '{comentario}'")
    except Exception as e:
        print(f"Aviso: Não foi possível comentar a coluna {coluna}. Erro: {e}")

print("✅ Governança aplicada: Dicionário de dados registrado no catálogo.")